In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Lodhi_Road_Delhi_IMD_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,190.0,111.0,131.0,59.0,67.0,59.0,NaN,NaN,123.0,122.0,337.0,346.0
1,2,284.0,111.0,160.0,65.0,55.0,78.0,46.0,NaN,124.0,115.0,319.0,325.0
2,3,346.0,116.0,111.0,93.0,77.0,89.0,88.0,66.0,116.0,119.0,458.0,257.0
3,4,273.0,154.0,106.0,79.0,53.0,110.0,116.0,83.0,119.0,129.0,362.0,272.0
4,5,274.0,156.0,108.0,84.0,107.0,122.0,70.0,70.0,104.0,141.0,419.0,NaN
5,6,365.0,170.0,109.0,93.0,175.0,85.0,55.0,79.0,86.0,151.0,369.0,220.0
6,7,332.0,229.0,110.0,95.0,113.0,230.0,NaN,96.0,NaN,137.0,348.0,219.0
7,8,342.0,101.0,160.0,112.0,108.0,126.0,51.0,104.0,NaN,122.0,377.0,235.0
8,9,406.0,131.0,85.0,120.0,145.0,NaN,57.0,115.0,NaN,134.0,400.0,240.0
9,10,370.0,135.0,117.0,100.0,153.0,119.0,44.0,132.0,NaN,124.0,244.0,264.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    34 non-null     float64
 2   February   32 non-null     float64
 3   March      36 non-null     float64
 4   April      33 non-null     float64
 5   May        35 non-null     float64
 6   June       28 non-null     float64
 7   July       28 non-null     float64
 8   August     29 non-null     float64
 9   September  26 non-null     float64
 10  October    34 non-null     float64
 11  November   35 non-null     float64
 12  December   32 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,190.0,111.0,131.0,59.0,67.0,59.000000,55.714286,93.517241,81.961538,122.0,337.0,346.00000
1,2,284.0,111.0,160.0,65.0,55.0,78.000000,46.000000,93.517241,81.961538,115.0,319.0,325.00000
2,3,346.0,116.0,111.0,93.0,77.0,89.000000,55.714286,66.000000,116.000000,119.0,458.0,257.00000
3,4,273.0,154.0,106.0,79.0,53.0,110.000000,55.714286,83.000000,119.000000,129.0,362.0,272.00000
4,5,274.0,156.0,108.0,84.0,107.0,81.785714,70.000000,70.000000,104.000000,141.0,419.0,257.34375


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
